# Building Segmentation — Training Notebook (Kaggle)

This notebook is **training only** and is designed for the dataset produced by the previous OpenSR notebook.

## Input

Upload or attach:

`sr_training_30.zip`

The ZIP should contain the 30 georeferenced super-resolved GeoTIFFs and `manifest.csv`.

Expected SR band order:

- Band 1 = B04 = Red
- Band 2 = B03 = Green
- Band 3 = B02 = Blue
- Band 4 = B08 = NIR

The U-Net uses **bands 1–3 (RGB)**.

## Labels

No mask dataset is required.

The notebook automatically downloads OpenStreetMap building footprints for each image and rasterizes them into binary training masks.

**Kaggle Internet must be enabled.**

## Outputs

- `/kaggle/working/unet_buildings.keras`
- `/kaggle/working/segmentation_training_config.json`
- `/kaggle/working/footprint_cache/`

After training, these model files will be used by the inference stage.


In [ ]:
!pip install -q osmnx

In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import json
import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf

import rasterio
from rasterio.features import rasterize

import geopandas as gpd
import osmnx as ox

from shapely.geometry import box
from pyproj import Transformer

print("TensorFlow:", tf.__version__)
print("Rasterio:", rasterio.__version__)
print("GeoPandas:", gpd.__version__)
print("OSMnx:", ox.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

## 1. Locate and extract the SR training ZIP

Attach `sr_training_30.zip` to the notebook using **Add Input**.

The next cell automatically finds it under `/kaggle/input` and extracts it to `/kaggle/working/sr_training_data`.


In [ ]:
# ============================================================
# Locate + extract sr_training_30.zip
# ============================================================
import os
import glob
import zipfile
import shutil

ZIP_NAME = "sr_training_30.zip"
EXTRACT_DIR = "/kaggle/working/sr_training_data"

zip_matches = glob.glob(
    f"/kaggle/input/**/{ZIP_NAME}",
    recursive=True
)

if not zip_matches:
    print("Could not find:", ZIP_NAME)
    print("\nZIP files currently attached to this Kaggle notebook:")
    all_zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)

    for z in all_zips:
        print(" -", z)

    raise FileNotFoundError(
        f"Attach {ZIP_NAME} using Kaggle → Add Input, then rerun."
    )

ZIP_PATH = zip_matches[0]

print("ZIP found:")
print(ZIP_PATH)

# Clean previous extraction so reruns are deterministic.
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("\nExtracted to:")
print(EXTRACT_DIR)

# Check the contents.
tifs = sorted(
    glob.glob(
        os.path.join(EXTRACT_DIR, "**", "*.tif"),
        recursive=True
    )
)

print(f"\nSR GeoTIFFs found: {len(tifs)}")

for p in tifs[:10]:
    print(" -", p)

if len(tifs) == 0:
    raise RuntimeError(
        "The ZIP was extracted, but no .tif files were found."
    )

manifest_matches = glob.glob(
    os.path.join(EXTRACT_DIR, "**", "manifest.csv"),
    recursive=True
)

if manifest_matches:
    print("\nManifest found:")
    print(manifest_matches[0])
else:
    print("\n⚠️ No manifest.csv found. Training can still continue.")


## 2. Training configuration

The dataset has already been extracted by the previous cell.

The training notebook will:

1. read the SR GeoTIFFs;
2. use RGB bands B04/B03/B02;
3. fetch OpenStreetMap building footprints;
4. rasterize them into masks;
5. split images into train/validation sets;
6. sample balanced 256×256 patches;
7. train a U-Net;
8. select the best probability threshold;
9. save the trained model and configuration.


In [ ]:
# ============================================================
# Configuration
# ============================================================

DATASET_PATH = EXTRACT_DIR
OUTPUT_DIR = "/kaggle/working"
CACHE_DIR = os.path.join(OUTPUT_DIR, "footprint_cache")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# OpenSR output:
# Band 1 = B04 Red
# Band 2 = B03 Green
# Band 3 = B02 Blue
# Band 4 = B08 NIR
BANDS = [1, 2, 3]

# Label generation
BUFFER_DEG = 0.005
RASTERIZE_ALL_TOUCHED = False
LABEL_DILATE_PIXELS = 0

# Patch sampling
PATCH_SIZE = 256
N_PATCHES_PER_IMAGE = 60
POS_FRACTION = 0.60
MIN_POS_RATIO = 0.01
MAX_NEG_RATIO = 0.001

# Image-level validation split
VAL_IMAGE_FRACTION = 0.20

# Training
BATCH_SIZE = 16
EPOCHS = 40
LR = 1e-4
SEED = 42

MODEL_PATH = os.path.join(OUTPUT_DIR, "unet_buildings.keras")
CONFIG_PATH = os.path.join(
    OUTPUT_DIR,
    "segmentation_training_config.json"
)

np.random.seed(SEED)
tf.random.set_seed(SEED)

all_tif_files = sorted(
    glob.glob(
        os.path.join(DATASET_PATH, "**", "*.tif"),
        recursive=True
    )
)

print(f"Found {len(all_tif_files)} SR GeoTIFF files in extracted dataset")

for path in all_tif_files[:10]:
    print(" -", path)

if len(all_tif_files) < 2:
    raise RuntimeError(
        "Training needs at least 2 georeferenced SR GeoTIFFs so train "
        "and validation images can be separated. For a useful model, "
        "use many more images."
    )

## 3. Read and validate the SR GeoTIFFs

In [ ]:
def percentile_normalize(img, pmin=2, pmax=98, eps=1e-6):
    """Robust per-band normalization to [0, 1]."""
    img = img.astype(np.float32)
    out = np.zeros_like(img, dtype=np.float32)

    for c in range(img.shape[2]):
        band = img[..., c]
        lo = np.percentile(band, pmin)
        hi = np.percentile(band, pmax)

        if hi - lo < eps:
            out[..., c] = 0.0
        else:
            out[..., c] = (band - lo) / (hi - lo)

    return np.clip(out, 0.0, 1.0)


def load_geotiff_rgb(path, bands=BANDS):
    """
    Returns:
        img       : (H, W, 3), float32 in [0,1]
        profile   : rasterio profile
        crs       : image CRS
        transform : affine transform
        bounds    : raster bounds
    """
    with rasterio.open(path) as src:
        if src.count < max(bands):
            raise ValueError(
                f"{path} has {src.count} band(s), but bands {bands} "
                "were requested."
            )

        if src.crs is None:
            raise ValueError(
                f"{path} has no CRS. Training labels cannot be aligned."
            )

        profile = src.profile.copy()
        crs = src.crs
        transform = src.transform
        bounds = src.bounds

        arr = np.stack(
            [src.read(b) for b in bands],
            axis=-1
        )

    img = percentile_normalize(arr)
    return img, profile, crs, transform, bounds


n_show = min(3, len(all_tif_files))

plt.figure(figsize=(6 * n_show, 6))

for i, path in enumerate(all_tif_files[:n_show]):
    img, profile, crs, transform, bounds = load_geotiff_rgb(path)

    print(
        f"{os.path.basename(path)} | "
        f"shape={img.shape} | CRS={crs} | "
        f"resolution=({abs(transform.a):.3f}, "
        f"{abs(transform.e):.3f})"
    )

    plt.subplot(1, n_show, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(path))
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Download OSM building footprints and create binary masks

The georeferencing of each SR image is used to request OpenStreetMap building polygons for the corresponding area. Those polygons are reprojected to the image CRS and rasterized onto the exact same pixel grid as the SR image.

In [ ]:
def bounds_to_wgs84(bounds, src_crs):
    transformer = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True
    )

    lon1, lat1 = transformer.transform(
        bounds.left,
        bounds.bottom
    )
    lon2, lat2 = transformer.transform(
        bounds.right,
        bounds.top
    )

    west, east = min(lon1, lon2), max(lon1, lon2)
    south, north = min(lat1, lat2), max(lat1, lat2)

    return west, south, east, north


def download_osm_buildings(
    bounds,
    src_crs,
    buffer_deg=BUFFER_DEG
):
    west, south, east, north = bounds_to_wgs84(
        bounds,
        src_crs
    )

    bbox_poly = box(
        west - buffer_deg,
        south - buffer_deg,
        east + buffer_deg,
        north + buffer_deg
    )

    img_poly_src = box(
        bounds.left,
        bounds.bottom,
        bounds.right,
        bounds.top
    )

    gdf = ox.features_from_polygon(
        bbox_poly,
        tags={"building": True}
    )

    if len(gdf) == 0:
        return gpd.GeoDataFrame(
            {"geometry": []},
            crs=src_crs
        )

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    gdf = gdf.to_crs(src_crs)

    gdf = gdf[
        gdf.geometry.notnull()
        & gdf.geometry.is_valid
        & gdf.geometry.geom_type.isin(
            ["Polygon", "MultiPolygon"]
        )
    ].copy()

    gdf = gdf[
        gdf.intersects(img_poly_src)
    ].copy()

    return gdf


def rasterize_buildings(
    footprints_gdf,
    out_shape,
    transform
):
    if len(footprints_gdf) == 0:
        return np.zeros(
            out_shape,
            dtype=np.uint8
        )

    shapes = [
        (geom, 1)
        for geom in footprints_gdf.geometry
    ]

    return rasterize(
        shapes=shapes,
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype=np.uint8,
        all_touched=RASTERIZE_ALL_TOUCHED
    )


def footprint_cache_path(image_path):
    token = abs(hash(str(image_path))) % 10_000_000
    stem = Path(image_path).stem

    return os.path.join(
        CACHE_DIR,
        f"{stem}_{token}_footprints.geojson"
    )


images = []
masks = []
metadata = []

print(
    f"Creating labels for "
    f"{len(all_tif_files)} images...\n"
)

for i, path in enumerate(all_tif_files, start=1):
    print(
        f"[{i}/{len(all_tif_files)}] "
        f"{os.path.basename(path)}"
    )

    try:
        img, profile, crs, transform, bounds = (
            load_geotiff_rgb(path)
        )

        cache_path = footprint_cache_path(path)

        if os.path.exists(cache_path):
            footprints = (
                gpd.read_file(cache_path)
                .to_crs(crs)
            )
            print(
                "  Loaded cached OSM footprints:",
                len(footprints)
            )
        else:
            footprints = download_osm_buildings(
                bounds,
                crs
            )

            print(
                "  OSM footprints found:",
                len(footprints)
            )

            if len(footprints) > 0:
                footprints.to_file(
                    cache_path,
                    driver="GeoJSON"
                )

        mask = rasterize_buildings(
            footprints,
            out_shape=img.shape[:2],
            transform=transform
        )

        if LABEL_DILATE_PIXELS > 0:
            k = 2 * LABEL_DILATE_PIXELS + 1

            kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,
                (k, k)
            )

            mask = cv2.dilate(
                mask,
                kernel,
                iterations=1
            )

        if mask.sum() == 0:
            print(
                "  Skipped: no building pixels "
                "in this image."
            )
            continue

        images.append(img)
        masks.append(mask)

        metadata.append(
            {
                "name": os.path.basename(path),
                "path": path,
                "crs": str(crs),
                "transform": tuple(transform)
            }
        )

        print(
            f"  Building coverage: "
            f"{100.0 * mask.mean():.2f}%"
        )

    except Exception as e:
        print("  ERROR:", e)


print(
    f"\nUsable labelled images: "
    f"{len(images)}"
)

if len(images) < 2:
    raise RuntimeError(
        "Fewer than 2 labelled images remain. "
        "Add more SR images or check that OSM "
        "contains footprints for those areas."
    )

In [ ]:
# Visual check of SR image / OSM mask alignment

n_show = min(4, len(images))
plt.figure(figsize=(15, 5 * n_show))

for i in range(n_show):
    img = images[i]
    mask = masks[i]

    plt.subplot(n_show, 3, 3*i + 1)
    plt.imshow(img)
    plt.title(metadata[i]["name"])
    plt.axis("off")

    plt.subplot(n_show, 3, 3*i + 2)
    plt.imshow(mask, cmap="gray")
    plt.title("OSM building mask")
    plt.axis("off")

    plt.subplot(n_show, 3, 3*i + 3)
    plt.imshow(img)
    plt.imshow(mask, cmap="Reds", alpha=0.35)
    plt.title("Alignment check")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 5. Split by image, then create balanced training patches

The split is done at **image level**, not after patch extraction. This avoids patches from the same satellite image appearing in both the training and validation sets.

In [ ]:
rng = np.random.default_rng(SEED)

indices = np.arange(len(images))
rng.shuffle(indices)

n_val_images = max(
    1,
    int(round(
        len(images) * VAL_IMAGE_FRACTION
    ))
)

n_val_images = min(
    n_val_images,
    len(images) - 1
)

val_image_idx = indices[:n_val_images]
train_image_idx = indices[n_val_images:]

print("Training images:")
for i in train_image_idx:
    print(" -", metadata[i]["name"])

print("\nValidation images:")
for i in val_image_idx:
    print(" -", metadata[i]["name"])

In [ ]:
def sample_patches_balanced(
    img,
    mask,
    patch_size,
    n_patches,
    pos_fraction=0.6,
    min_pos_ratio=0.01,
    max_neg_ratio=0.001,
    max_tries=200000,
    seed=42
):
    rng = np.random.default_rng(seed)

    H, W = mask.shape
    ps = patch_size

    if H < ps or W < ps:
        return (
            np.empty(
                (0, ps, ps, 3),
                dtype=np.float32
            ),
            np.empty(
                (0, ps, ps, 1),
                dtype=np.float32
            )
        )

    ys, xs = np.where(mask > 0)
    has_pos = len(xs) > 0

    X, Y = [], []

    n_pos = int(
        round(n_patches * pos_fraction)
    )
    n_neg = n_patches - n_pos

    def clip_xy(cx, cy):
        x0 = int(
            np.clip(
                cx - ps // 2,
                0,
                W - ps
            )
        )
        y0 = int(
            np.clip(
                cy - ps // 2,
                0,
                H - ps
            )
        )
        return x0, y0

    tries = 0

    while (
        len(X) < n_pos
        and tries < max_tries
    ):
        tries += 1

        if not has_pos:
            break

        j = rng.integers(0, len(xs))
        cx, cy = xs[j], ys[j]

        x0, y0 = clip_xy(cx, cy)

        pm = mask[
            y0:y0+ps,
            x0:x0+ps
        ]

        if pm.mean() >= min_pos_ratio:
            X.append(
                img[
                    y0:y0+ps,
                    x0:x0+ps
                ]
            )
            Y.append(pm[..., None])

    neg_count = 0
    tries = 0

    while (
        neg_count < n_neg
        and tries < max_tries
    ):
        tries += 1

        x0 = rng.integers(
            0,
            W - ps + 1
        )
        y0 = rng.integers(
            0,
            H - ps + 1
        )

        pm = mask[
            y0:y0+ps,
            x0:x0+ps
        ]

        if pm.mean() <= max_neg_ratio:
            X.append(
                img[
                    y0:y0+ps,
                    x0:x0+ps
                ]
            )
            Y.append(pm[..., None])
            neg_count += 1

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


def patches_from_image_indices(
    image_indices,
    seed_offset=0
):
    X_parts = []
    Y_parts = []

    for k, i in enumerate(image_indices):
        xp, yp = sample_patches_balanced(
            images[i],
            masks[i],
            patch_size=PATCH_SIZE,
            n_patches=N_PATCHES_PER_IMAGE,
            pos_fraction=POS_FRACTION,
            min_pos_ratio=MIN_POS_RATIO,
            max_neg_ratio=MAX_NEG_RATIO,
            seed=SEED + seed_offset + k
        )

        if len(xp) > 0:
            X_parts.append(xp)
            Y_parts.append(yp)

    if not X_parts:
        raise RuntimeError(
            "No patches could be generated."
        )

    return (
        np.concatenate(X_parts),
        np.concatenate(Y_parts)
    )


X_train, Y_train = (
    patches_from_image_indices(
        train_image_idx,
        seed_offset=0
    )
)

X_val, Y_val = (
    patches_from_image_indices(
        val_image_idx,
        seed_offset=10000
    )
)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:  ", X_val.shape)
print("Y_val:  ", Y_val.shape)

In [ ]:
# Patch sanity check

n_show = min(4, len(X_train))

plt.figure(figsize=(12, 6))

for i in range(n_show):
    j = np.random.randint(
        0,
        len(X_train)
    )

    plt.subplot(
        2,
        n_show,
        i + 1
    )
    plt.imshow(X_train[j])
    plt.axis("off")
    plt.title("Image")

    plt.subplot(
        2,
        n_show,
        n_show + i + 1
    )
    plt.imshow(
        Y_train[j, ..., 0],
        cmap="gray"
    )
    plt.axis("off")
    plt.title("Mask")

plt.tight_layout()
plt.show()

## 6. TensorFlow input pipeline and data augmentation

In [ ]:
def augment_pair(x, y):
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_left_right(x)
        y = tf.image.flip_left_right(y)

    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_up_down(x)
        y = tf.image.flip_up_down(y)

    k = tf.random.uniform(
        (),
        minval=0,
        maxval=4,
        dtype=tf.int32
    )

    x = tf.image.rot90(x, k)
    y = tf.image.rot90(y, k)

    x = tf.image.random_brightness(
        x,
        max_delta=0.1
    )

    x = tf.image.random_contrast(
        x,
        lower=0.8,
        upper=1.2
    )

    x = tf.clip_by_value(
        x,
        0.0,
        1.0
    )

    return x, y


def make_dataset(
    X,
    Y,
    batch_size,
    training
):
    ds = tf.data.Dataset.from_tensor_slices(
        (X, Y)
    )

    if training:
        ds = ds.shuffle(
            len(X),
            seed=SEED,
            reshuffle_each_iteration=True
        )

        ds = ds.map(
            augment_pair,
            num_parallel_calls=tf.data.AUTOTUNE
        )

    return (
        ds.batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


train_ds = make_dataset(
    X_train,
    Y_train,
    BATCH_SIZE,
    training=True
)

val_ds = make_dataset(
    X_val,
    Y_val,
    BATCH_SIZE,
    training=False
)

print("TensorFlow datasets ready.")

## 7. U-Net model

In [ ]:
from tensorflow.keras import layers, Model


def conv_block(x, filters, dropout=0.0):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    return x


def build_unet(
    input_shape=(256, 256, 3),
    base=32
):
    inp = layers.Input(
        shape=input_shape
    )

    c1 = conv_block(inp, base)
    p1 = layers.MaxPool2D()(c1)

    c2 = conv_block(p1, base * 2)
    p2 = layers.MaxPool2D()(c2)

    c3 = conv_block(
        p2,
        base * 4,
        dropout=0.1
    )
    p3 = layers.MaxPool2D()(c3)

    c4 = conv_block(
        p3,
        base * 8,
        dropout=0.2
    )
    p4 = layers.MaxPool2D()(c4)

    b = conv_block(
        p4,
        base * 16,
        dropout=0.3
    )

    u4 = layers.UpSampling2D()(b)
    u4 = layers.Concatenate()(
        [u4, c4]
    )
    c5 = conv_block(
        u4,
        base * 8,
        dropout=0.2
    )

    u3 = layers.UpSampling2D()(c5)
    u3 = layers.Concatenate()(
        [u3, c3]
    )
    c6 = conv_block(
        u3,
        base * 4,
        dropout=0.1
    )

    u2 = layers.UpSampling2D()(c6)
    u2 = layers.Concatenate()(
        [u2, c2]
    )
    c7 = conv_block(
        u2,
        base * 2
    )

    u1 = layers.UpSampling2D()(c7)
    u1 = layers.Concatenate()(
        [u1, c1]
    )
    c8 = conv_block(
        u1,
        base
    )

    out = layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c8)

    return Model(inp, out)


def dice_coef(
    y_true,
    y_pred,
    smooth=1e-6
):
    y_true = tf.cast(
        y_true,
        tf.float32
    )
    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    y_true_f = tf.reshape(
        y_true,
        [-1]
    )
    y_pred_f = tf.reshape(
        y_pred,
        [-1]
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    denominator = (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
    )

    return (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )


def bce_dice_loss(
    y_true,
    y_pred
):
    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(bce)

    return (
        bce
        + (1.0 - dice_coef(
            y_true,
            y_pred
        ))
    )


strategy = tf.distribute.MirroredStrategy()

print(
    "Training replicas:",
    strategy.num_replicas_in_sync
)

with strategy.scope():
    model = build_unet(
        input_shape=(
            PATCH_SIZE,
            PATCH_SIZE,
            3
        ),
        base=32
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LR
        ),
        loss=bce_dice_loss,
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.BinaryIoU(
                target_class_ids=[1],
                threshold=0.5,
                name="building_iou"
            ),
            dice_coef
        ]
    )

model.summary()

## 8. Train and save the best model

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=4,
        factor=0.5,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(
    history.history["loss"],
    label="Train"
)
plt.plot(
    history.history["val_loss"],
    label="Validation"
)
plt.title("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(
    history.history["building_iou"],
    label="Train"
)
plt.plot(
    history.history["val_building_iou"],
    label="Validation"
)
plt.title("Building IoU")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(
    history.history["dice_coef"],
    label="Train"
)
plt.plot(
    history.history["val_dice_coef"],
    label="Validation"
)
plt.title("Dice")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Tune the probability threshold on validation data

In [ ]:
def compute_metrics_binary(
    y_true,
    y_pred_bin
):
    y_true = (
        y_true > 0.5
    ).astype(
        np.uint8
    ).reshape(-1)

    y_pred = (
        y_pred_bin > 0
    ).astype(
        np.uint8
    ).reshape(-1)

    tp = np.sum(
        (y_true == 1)
        & (y_pred == 1)
    )

    fp = np.sum(
        (y_true == 0)
        & (y_pred == 1)
    )

    fn = np.sum(
        (y_true == 1)
        & (y_pred == 0)
    )

    tn = np.sum(
        (y_true == 0)
        & (y_pred == 0)
    )

    precision = (
        tp / (tp + fp + 1e-8)
    )

    recall = (
        tp / (tp + fn + 1e-8)
    )

    f1 = (
        2 * precision * recall
        / (
            precision
            + recall
            + 1e-8
        )
    )

    iou = (
        tp
        / (
            tp
            + fp
            + fn
            + 1e-8
        )
    )

    dice = (
        2 * tp
        / (
            2 * tp
            + fp
            + fn
            + 1e-8
        )
    )

    return {
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "iou": float(iou),
        "dice": float(dice)
    }


val_probs = model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
)

best_threshold = 0.5
best_metrics = None

for threshold in np.linspace(
    0.20,
    0.80,
    13
):
    metrics = compute_metrics_binary(
        Y_val,
        (
            val_probs >= threshold
        ).astype(np.uint8)
    )

    print(
        f"threshold={threshold:.2f} | "
        f"F1={metrics['f1']:.4f} | "
        f"IoU={metrics['iou']:.4f} | "
        f"precision={metrics['precision']:.4f} | "
        f"recall={metrics['recall']:.4f}"
    )

    if (
        best_metrics is None
        or metrics["f1"] > best_metrics["f1"]
    ):
        best_threshold = float(
            threshold
        )
        best_metrics = metrics


print(
    "\nBest threshold:",
    best_threshold
)

print(
    "Best validation metrics:",
    best_metrics
)

## 10. Save metadata needed by the inference notebook

In [ ]:
training_config = {
    "model_path": MODEL_PATH,

    "input_bands_1_based": BANDS,

    "band_meaning": {
        "1": "B04 / Red",
        "2": "B03 / Green",
        "3": "B02 / Blue"
    },

    "normalization": (
        "per-image per-band percentile "
        "normalization p2-p98 to [0,1]"
    ),

    "patch_size": PATCH_SIZE,

    "recommended_stride": (
        PATCH_SIZE // 2
    ),

    "prediction_threshold": (
        best_threshold
    ),

    "validation_metrics_at_selected_threshold": (
        best_metrics
    ),

    "train_images": [
        metadata[i]["name"]
        for i in train_image_idx
    ],

    "validation_images": [
        metadata[i]["name"]
        for i in val_image_idx
    ],

    "label_source": (
        "OpenStreetMap building footprints"
    ),

    "seed": SEED
}

with open(
    CONFIG_PATH,
    "w"
) as f:
    json.dump(
        training_config,
        f,
        indent=2
    )

print("Saved model:")
print(MODEL_PATH)

print(
    "\nSaved training/inference config:"
)
print(CONFIG_PATH)

# Training finished

The next segmentation/inference notebook needs:

1. `unet_buildings.keras`
2. `segmentation_training_config.json`
3. a new `sr.tif` from the super-resolution stage

It should load the model with `compile=False`, apply the same RGB-band selection and percentile normalization, perform tiled inference, apply the saved threshold, and write a georeferenced `building_mask.tif`.

That mask is then passed to the PV-estimation stage.